In [1]:
import json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TFAutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
import tensorflow as tf

/Users/keertinayak30/.conda/envs/here-nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Loading intent classifier...")
tokenizer = AutoTokenizer.from_pretrained("intent_model_v2_pt")
# bert_model = AutoModelForSequenceClassification.from_pretrained("intent_model_v2_pt")
# bert_model.eval()
bert_model = TFAutoModelForSequenceClassification.from_pretrained("intent_model_v2")

Loading intent classifier...


2026-06-04 22:19:23.025735: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-06-04 22:19:23.025759: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-06-04 22:19:23.025765: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-06-04 22:19:23.025865: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-06-04 22:19:23.026145: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
All model checkpoint layers were used when initializing TFBertForSequenceClassification.

All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at 

In [3]:

# Load label map
with open("label_map.json") as f:
    label_map = json.load(f)
# label_map keys are strings ("0","1"...) — convert to int
label_map = {int(k): v for k, v in label_map.items()}
print("Labels:", label_map)

Labels: {0: 'atm', 1: 'cafe', 2: 'fuel', 3: 'hospital', 4: 'other', 5: 'parking', 6: 'pharmacy', 7: 'restaurant'}


In [4]:
# Load places
with open("places.json") as f:
    places = json.load(f)
print(f"Loaded {len(places)} places.")

Loaded 1831 places.


In [5]:
print("Embedding places...")
embedder = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

Embedding places...


In [6]:
CATEGORY_HINTS = {
    "pharmacy":   "medicine healthcare drugs medical store prescriptions",
    "restaurant": "food dining meals lunch dinner snacks eating",
    "atm":        "cash withdrawal money banking finance",
    "parking":    "vehicle parking car bike parking lot",
    "hospital":   "emergency doctor healthcare treatment medical",
    "fuel":       "petrol diesel pump filling station",
    "cafe":       "coffee tea snacks beverages drinks"
}

place_texts = [
    f"{p['name']} {p['category']} {p['desc']} {CATEGORY_HINTS.get(p['category'].lower(), '')}"
    for p in places
]
place_vectors = embedder.encode(place_texts, convert_to_numpy=True, normalize_embeddings=True)
place_vectors = np.nan_to_num(place_vectors)
print("Ready.")

Ready.


In [7]:
def predict_intent(query):
    inputs = tokenizer(
        query,
        return_tensors="tf",
        truncation=True,
        padding=True,
        max_length=32
    )
    logits = bert_model(inputs).logits
    predicted_id = int(tf.argmax(logits, axis=1)[0])
    return label_map[predicted_id]

In [8]:
def search_place(query, intent_category, top_k=3, threshold=0.15):
    query_vector = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Base semantic scores
    scores = place_vectors @ query_vector[0]
    
    # Soft boost — 0.15 so semantic search still does heavy lifting
    # but intent nudges ranking toward the right category
    if intent_category and intent_category != "other":
        for i, place in enumerate(places):
            if place["category"].lower() == intent_category:
                scores[i] += 0.15

    top_indices = np.argsort(scores)[::-1]
    results = []
    seen = set()

    for i in top_indices:
        name = places[i]["name"]
        if name in seen:
            continue
        if scores[i] >= threshold:
            seen.add(name)
            results.append({
                "name": name,
                "category": places[i]["category"],
                "score": round(float(scores[i]), 3)
            })
        if len(results) == top_k:
            break

    return results

In [9]:
def run(query):
    print(f"\nQuery    : {query}")
    
    intent = predict_intent(query)
    print(f"Intent   : {intent}")
    
    results = search_place(query, intent)
    print("Results  :")
    for r in results:
        print(f"  {r['name']} ({r['category']}) — {r['score']}")
    
    return intent, results

In [10]:
if __name__ == "__main__":
    test_queries = [
        "petrol pump chahiye",
        "doctor open now",
        "cash nikalna hai nearby",
        "where can i get coffee",
        "need medicines urgently",
        "parking near bandra station",
        "khana khane ki jagah",
        "hospital near juhu",
        "find south indian food",
        "nearest ATM for cash"
    ]
    
    for q in test_queries:
        run(q)


Query    : petrol pump chahiye
Intent   : fuel
Results  :
  Petrol Pump (fuel) — 0.78
  Zojwala Petroleum (fuel) — 0.763
  Indian Oil Petrol Pump (fuel) — 0.753

Query    : doctor open now
Intent   : hospital
Results  :
  Dr Gadgil Eye Clinic and Lasik Laser Centre (hospital) — 0.609
  Infinity Medisurge Centre Speciality Hospital (hospital) — 0.589
  Dr. Vaidya Eye Hospital (hospital) — 0.585

Query    : cash nikalna hai nearby
Intent   : atm
Results  :
  Kotak Mahindra ATM Juhu Tara road (atm) — 0.649
  State Bamk ATM Chheda Nagar (atm) — 0.643
  Yes Bank ATM - Juhu Tara Road (atm) — 0.64

Query    : where can i get coffee
Intent   : cafe
Results  :
  Bean Theory Coffee (cafe) — 0.661
  My Little Tea Pot (cafe) — 0.634
  My space (cafe) — 0.633

Query    : need medicines urgently
Intent   : hospital
Results  :
  Alphine life solutions general hospital (hospital) — 0.503
  Kokilaben Dhirubhai Ambani Hospital (hospital) — 0.49
  Dr Bhatia’s Hospital, Bhandup (hospital) — 0.489

Query 